## Import libraries

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import config
from IPython.display import display
from tqdm.auto import tqdm

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import ParameterSampler
from xgboost import XGBClassifier

## Functions

In [ ]:
def create_model(params=None):
    if params is None:
        params = best_xgb_params  # Defined after randomized search

    model_params = {
        **params,
        "reg_alpha": 1.0,
        "tree_method": "hist",
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "n_jobs": -1,
        "random_state": config.RANDOM_STATE,
    }

    return XGBClassifier(**model_params)

def X(data, columns):
    return data.loc[:, columns].replace([np.inf, -np.inf], np.nan)

def safe_auc(y, p):
    return float(roc_auc_score(y, p)) if np.unique(y).size == 2 else np.nan

def safe_ap(y, p):
    return float(average_precision_score(y, p)) if np.unique(y).size == 2 else np.nan

def direction_metrics(data, threshold):
    y = data[TARGET].to_numpy(dtype=int)
    p = data["Probability"].to_numpy(dtype=float)
    predicted_positive = p >= threshold
    selected = int(predicted_positive.sum())
    true_positive = int(np.sum(predicted_positive & (y == 1)))
    false_positive = int(np.sum(predicted_positive & (y == 0)))
    true_negative = int(np.sum(~predicted_positive & (y == 0)))
    false_negative = int(np.sum(~predicted_positive & (y == 1)))
    precision = true_positive / selected if selected else np.nan
    base = float(y.mean())
    return {
        "Observations": len(y),
        "Always_Positive_Precision": base,
        "Precision": precision,
        "Precision_Lift_pp": 100.0 * (precision - base) if selected else np.nan,
        "Recall": true_positive / (true_positive + false_negative) if np.any(y == 1) else np.nan,
        "Specificity": true_negative / (true_negative + false_positive) if np.any(y == 0) else np.nan,
        "Balanced_Accuracy": float(balanced_accuracy_score(y, predicted_positive)),
        "Positive_Prediction_Rate": selected / len(y),
        "Predicted_Positive_Count": selected,
        "ROC_AUC": safe_auc(y, p),
        "Average_Precision": safe_ap(y, p),
    }

def screening_summary(part):
    n_selected = int(part["Selected_Assets"].sum())
    n_correct = int(part["Correct_Positives"].sum())
    total_dates = len(part)
    n_active = int((part["Selected_Assets"] > 0).sum())
    precision = n_correct / n_selected if n_selected else np.nan
    matched_baseline = (
        float((part["Selected_Assets"] * part["Same_Date_Baseline"]).sum() / n_selected)
        if n_selected else np.nan
    )
    return {
        "Dates": total_dates,
        "Selected_Asset_Dates": n_selected,
        "Precision": precision,
        "Matched_Date_Baseline": matched_baseline,
        "Precision_Lift_pp": 100 * (precision - matched_baseline) if n_selected else np.nan,
        "Mean_Assets": n_selected / total_dates,
        "Min_Assets": int(part["Selected_Assets"].min()),
        "Max_Assets": int(part["Selected_Assets"].max()),
        "Cash_Only_Dates": total_dates - n_active,
    }

## Load features and define feature sets

In [2]:
# Set features folder
FEATURES_DATA_DIR = config.INPUT_DATA_DIR.parent / "features_data"

# Load asset classes from the investable universe
universe = pd.read_csv(config.UNIVERSE_FILE)

# Read features files
df = pd.read_csv(FEATURES_DATA_DIR / "combined_features.csv", parse_dates=["Date"])
regimes = pd.read_csv(FEATURES_DATA_DIR / "market_regimes.csv", parse_dates=["Date"])

# Merge with confirmed regime label
df = df.merge(
    regimes[["Date", "Market_Regime"]],
    on="Date",
    how="left",
    validate="many_to_one"
)

df = df.sort_values(["Date", "Ticker"]).reset_index(drop=True)

# Identify dates with missing regime labels
missing_dates = (df.loc[df["Market_Regime"].isna(), "Date"].drop_duplicates().sort_values())
missing_count = df["Market_Regime"].isna().sum()

# Forward fill using the last available confirmed regime
df["Market_Regime"] = df["Market_Regime"].ffill()

# Print dates and assigned regime labels
filled_dates = (
    df.loc[df["Date"].isin(missing_dates), ["Date", "Market_Regime"]]
    .drop_duplicates()
    .sort_values("Date")
)

print(f"Forward-filled regime dates: {len(missing_dates)}")

# Final checks
print("\nDataset shape:", df.shape)
print("Missing regime labels:", df["Market_Regime"].isna().sum())
print("Duplicate asset-dates:", df.duplicated(["Date", "Ticker"]).sum())
print(df["Target_60d"].value_counts(normalize=True))

asset_classes = universe[["Ticker", "Asset Class"]].drop_duplicates(subset="Ticker")

# Merge asset classes into the modeling dataset
df = df.merge(
    asset_classes,
    on="Ticker",
    how="left",
    validate="many_to_one"
)

# Encode asset classes
asset_class_dummies = pd.get_dummies(df["Asset Class"], prefix="AssetClass", dtype=int)
df = pd.concat([df, asset_class_dummies], axis=1)
asset_class_features = asset_class_dummies.columns.tolist()

print("\nAsset-class features:")
print(asset_class_features)

# One-hot encode ticker identity
ticker_dummies = pd.get_dummies(df["Ticker"], prefix="Ticker", dtype=int)
df = pd.concat([df, ticker_dummies], axis=1)
ticker_features = ticker_dummies.columns.tolist()

# Define target label
TARGET = "Target_60d"

# Columns exclude from features
EXCLUDE = ["Date", "Ticker", "Forward_Return_60d", "Target_60d", "Market_Regime", "Asset Class"]

# Define feature set
base_features = [c for c in df.columns if c not in EXCLUDE and not c.startswith("Regime_") and not c.startswith("Ticker_")]

# Use one-hot encoding for confirmed regimes
regime_dummies = pd.get_dummies(df["Market_Regime"], prefix="Regime", dtype=int)
df = pd.concat([df, regime_dummies], axis=1)
regime_features = regime_dummies.columns.tolist()

# Regime specifications
features_without_regime = base_features
features_with_regime = base_features + regime_features

# Ticker-level specifications
features_with_tickers = base_features + ticker_features
features_with_tickers_regime = (base_features + ticker_features + regime_features)

assert not df.duplicated(["Date", "Ticker"]).any()
assert df["Asset Class"].notna().all()
assert df[TARGET].notna().any()
assert not df["Market_Regime"].isna().any()
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in features_with_tickers_regime)

SPECIFICATIONS = {
    "Without Regimes": features_with_tickers,
    "With Regimes": features_with_tickers_regime,
}

for name, features in SPECIFICATIONS.items():
    assert len(features) == len(set(features)), name
    assert not set(["Date", "Ticker", TARGET, "Forward_Return_60d"]).intersection(features)

Forward-filled regime dates: 36

Dataset shape: (215430, 256)
Missing regime labels: 0
Duplicate asset-dates: 0
Target_60d
1    0.615105
0    0.384895
Name: proportion, dtype: float64

Asset-class features:
['AssetClass_Commodity', 'AssetClass_Crypto', 'AssetClass_Equity', 'AssetClass_Fixed Income', 'AssetClass_Foreign Exchange', 'AssetClass_Volatility']


## Walk-forward validation

In [3]:
# Validation period and rebalance frequency
VALIDATION_START = pd.Timestamp(config.VALIDATION_START)
TEST_START = pd.Timestamp(config.TEST_START)
REBALANCE_STEP = config.REBALANCE_FREQUENCY

# Calculate when each asset's 60-session target becomes known
df = df.sort_values(["Ticker", "Date"], kind="stable").reset_index(drop=True).copy()
df["_Target_End_Date"] = (
    df.groupby("Ticker", sort=False)["Date"].shift(-config.FORWARD_HORIZON)
)

valid_target = (
    df[TARGET].isin([0, 1])
    & df["Forward_Return_60d"].notna()
    & df["_Target_End_Date"].notna()
)

# Use the dates already available in the merged dataframe
trading_dates = pd.DatetimeIndex(sorted(df["Date"].dropna().unique()))

# Anchor the schedule to the first 2022 rebalance date
test_anchor = trading_dates[trading_dates >= TEST_START][0]
anchor_pos = trading_dates.get_loc(test_anchor)

# Generate 60-session rebalance dates backwards from 2022
positions = np.arange(anchor_pos - REBALANCE_STEP, -1, -REBALANCE_STEP)

validation_dates = trading_dates[positions]
validation_dates = validation_dates[
    (validation_dates >= VALIDATION_START)
    & (validation_dates < TEST_START)
].sort_values()

# Prepare expanding-window training and validation datasets
walk_forward_splits = []
validation_summary = []
skipped_dates = []

for rebalance_date in validation_dates:

    train = df.loc[
        (df["Date"] < rebalance_date)
        & (df["_Target_End_Date"] < rebalance_date)
        & valid_target
    ].copy()

    validation = df.loc[
        df["Date"] == rebalance_date
    ].copy()

    # Require complete, known outcomes for the rebalance date
    known = (
        valid_target.loc[validation.index]
        & (validation["_Target_End_Date"] < TEST_START)
    )

    if validation.empty or not known.all():
        skipped_dates.append(rebalance_date)
        continue

    assert not train.empty, rebalance_date
    assert train[TARGET].nunique() == 2, rebalance_date
    assert train["_Target_End_Date"].max() < rebalance_date
    assert validation["Date"].nunique() == 1
    assert validation["_Target_End_Date"].max() < TEST_START

    walk_forward_splits.append((rebalance_date, train, validation))

    validation_summary.append({
        "Rebalance_Date": rebalance_date,
        "Train_Start": train["Date"].min(),
        "Train_End": train["Date"].max(),
        "Train_Obs": len(train),
        "Val_Obs": len(validation),
        "Val_Baseline": validation[TARGET].mean(),
    })

# Ensure that no asset is evaluated twice on the same date
validation_assets = pd.concat(
    [v[["Date", "Ticker"]] for _, _, v in walk_forward_splits],
    ignore_index=True
)

assert not validation_assets.duplicated(["Date", "Ticker"]).any()

print("Scheduled validation dates:", len(validation_dates))
print("Completed validation dates:", len(walk_forward_splits))
print("Excluded immature/incomplete dates:", len(skipped_dates))

display(pd.DataFrame(validation_summary))

Scheduled validation dates: 25
Completed validation dates: 24
Excluded immature/incomplete dates: 1


,Rebalance_Date,Train_Start,Train_End,Train_Obs,Val_Obs,Val_Baseline
0,2016-01-20,2008-01-02,2015-10-21,86587,48,0.958333
1,2016-04-15,2008-01-02,2016-01-19,89467,48,0.833333
2,2016-07-12,2008-01-02,2016-04-14,92347,48,0.541667
3,2016-10-05,2008-01-02,2016-07-11,95227,48,0.333333
4,2016-12-30,2008-01-02,2016-10-04,98107,48,0.895833
5,2017-03-29,2008-01-02,2016-12-29,100987,48,0.833333
6,2017-06-23,2008-01-02,2017-03-28,103867,48,0.875000
7,2017-09-19,2008-01-02,2017-06-22,106747,48,0.666667
8,2017-12-13,2008-01-02,2017-09-18,109627,48,0.625000
9,2018-03-13,2008-01-02,2017-12-12,112507,48,0.541667


## XGBoost and hyperparameter search space

In [4]:
ALGORITHMS = ["XGBoost"]
TUNING_SPECIFICATION = "With Regimes"
N_ITER = 30

XGB_PARAM_SPACE = {
    "max_depth": [1, 2, 3, 4],
    "n_estimators": [75, 100, 150, 250],
    "learning_rate": [0.02, 0.05, 0.10],
    "min_child_weight": [20, 50, 100],
    "reg_lambda": [1, 10, 20, 50],
    "subsample": [0.70, 0.85, 1.00],
    "colsample_bytree": [0.70, 0.85, 1.00],
}

parameter_sets = list(ParameterSampler(
    XGB_PARAM_SPACE,
    n_iter=N_ITER,
    random_state=config.RANDOM_STATE,
))


print(f"XGBoost parameter configurations: {len(parameter_sets)}")
print(f"Walk-forward validation dates: {len(walk_forward_splits)}")
print(f"Search model fits: {len(parameter_sets) * len(walk_forward_splits)}")

XGBoost parameter configurations: 30
Walk-forward validation dates: 24
Search model fits: 720


## 5. Randomized XGBoost hyperparameter search

In [5]:
tuning_features = SPECIFICATIONS[TUNING_SPECIFICATION]
search_daily_rows = []

with tqdm(
    total=len(parameter_sets) * len(walk_forward_splits),
    desc="XGBoost random search",
    unit="fit",
) as progress:
    for trial, params in enumerate(parameter_sets, start=1):
        for rebalance_date, train, validation in walk_forward_splits:
            y_train = train[TARGET].astype(int).to_numpy()
            y_val = validation[TARGET].astype(int).to_numpy()

            model = create_model(params=params)
            model.fit(X(train, tuning_features), y_train)

            p_train = model.predict_proba(X(train, tuning_features))[:, 1]
            p_val = model.predict_proba(X(validation, tuning_features))[:, 1]

            train_auc = safe_auc(y_train, p_train)
            val_auc = safe_auc(y_val, p_val)

            search_daily_rows.append({
                "Trial": trial,
                "Rebalance_Date": rebalance_date,
                "Train_Observations": len(y_train),
                "Validation_Observations": len(y_val),
                "Train_ROC_AUC": train_auc,
                "Val_ROC_AUC": val_auc,
                "Val_Average_Precision": safe_ap(y_val, p_val),
                "Val_Log_Loss": float(log_loss(y_val, p_val, labels=[0, 1])),
                "AUC_Gap": train_auc - val_auc,
            })

            progress.set_postfix(trial=trial, date=rebalance_date.strftime("%Y-%m-%d"))
            progress.update(1)

search_by_rebalance = pd.DataFrame(search_daily_rows)

parameter_table = pd.DataFrame([
    {"Trial": trial, **params, "Parameters_JSON": json.dumps(params, sort_keys=True)}
    for trial, params in enumerate(parameter_sets, start=1)
])

hyperparameter_search_results = (
    search_by_rebalance.groupby("Trial", as_index=False)
    .agg(
        Validation_Dates=("Rebalance_Date", "nunique"),
        Val_ROC_AUC_Mean=("Val_ROC_AUC", "mean"),
        Val_ROC_AUC_SD=("Val_ROC_AUC", "std"),
        Valid_AUC_Dates=("Val_ROC_AUC", "count"),
        Val_AP_Mean=("Val_Average_Precision", "mean"),
        Val_Log_Loss_Mean=("Val_Log_Loss", "mean"),
        Train_ROC_AUC_Mean=("Train_ROC_AUC", "mean"),
        AUC_Gap_Mean=("AUC_Gap", "mean"),
    )
    .merge(parameter_table, on="Trial", validate="one_to_one")
    .sort_values(
        ["Val_ROC_AUC_Mean", "Val_AP_Mean", "AUC_Gap_Mean"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

assert (hyperparameter_search_results["Validation_Dates"] == len(walk_forward_splits)).all()
assert hyperparameter_search_results["Valid_AUC_Dates"].nunique() == 1
assert hyperparameter_search_results["Val_ROC_AUC_Mean"].notna().all()

# Highest mean validation ROC AUC wins. AP and AUC gap break exact ties.
best_trial = int(hyperparameter_search_results.loc[0, "Trial"])
best_xgb_params = dict(parameter_sets[best_trial - 1])
hyperparameter_search_results["Selected"] = (
    hyperparameter_search_results["Trial"] == best_trial
)

best_hyperparameters = pd.DataFrame([{
    "Algorithm": "XGBoost",
    "Tuning_Specification": TUNING_SPECIFICATION,
    "Trial": best_trial,
    "Validation_Dates": len(walk_forward_splits),
    "Selection_Metric": "Mean within-date validation ROC AUC",
    "Val_ROC_AUC_Mean": hyperparameter_search_results.loc[0, "Val_ROC_AUC_Mean"],
    "Val_AP_Mean": hyperparameter_search_results.loc[0, "Val_AP_Mean"],
    **best_xgb_params,
}])

print("Selected randomized-search configuration:")
print(json.dumps(best_xgb_params, indent=2, sort_keys=True))
display(hyperparameter_search_results[[
    "Trial", "Selected", "Val_ROC_AUC_Mean", "Val_ROC_AUC_SD",
    "Val_AP_Mean", "Val_Log_Loss_Mean", "Train_ROC_AUC_Mean",
    "AUC_Gap_Mean", *XGB_PARAM_SPACE.keys(),
]].round(4))

XGBoost random search:   0%|          | 0/720 [00:00<?, ?fit/s]

Selected randomized-search configuration:
{
  "colsample_bytree": 0.7,
  "learning_rate": 0.05,
  "max_depth": 2,
  "min_child_weight": 100,
  "n_estimators": 250,
  "reg_lambda": 20,
  "subsample": 1.0
}


,Trial,Selected,Val_ROC_AUC_Mean,Val_ROC_AUC_SD,Val_AP_Mean,Val_Log_Loss_Mean,Train_ROC_AUC_Mean,AUC_Gap_Mean,max_depth,n_estimators,learning_rate,min_child_weight,reg_lambda,subsample,colsample_bytree
0,1,True,0.7014,0.1964,0.7579,0.6447,0.8294,0.1280,2,250,0.05,100,20,1.00,0.70
1,3,False,0.6771,0.2063,0.7350,0.6525,0.8239,0.1468,2,100,0.10,50,20,1.00,0.85
2,26,False,0.6730,0.2080,0.7425,0.6611,0.8359,0.1629,3,100,0.05,100,10,0.70,0.70
3,23,False,0.6673,0.2049,0.7301,0.6534,0.8023,0.1350,2,100,0.05,100,1,1.00,0.85
4,7,False,0.6652,0.2023,0.7426,0.6776,0.8540,0.1888,3,75,0.10,20,10,0.70,0.85
5,28,False,0.6651,0.2046,0.7324,0.6649,0.8003,0.1352,2,250,0.02,20,50,0.70,0.70
6,10,False,0.6637,0.2068,0.7474,0.6694,0.8814,0.2176,4,75,0.10,100,10,1.00,0.70
7,29,False,0.6623,0.2137,0.7415,0.6726,0.8651,0.2028,3,250,0.05,100,50,0.85,0.85
8,19,False,0.6618,0.2076,0.7481,0.6686,0.8551,0.1933,2,250,0.10,50,20,0.85,1.00
9,5,False,0.6601,0.2115,0.7376,0.6736,0.8263,0.1662,3,75,0.05,100,50,0.85,1.00


## 6. Retrain tuned XGBoost and generate validation probabilities

In [6]:
prediction_parts = []
fit_rows = []

for rebalance_date, train, validation in tqdm(
    walk_forward_splits,
    desc="Tuned XGBoost walk-forward",
    unit="rebalance",
):
    y_train = train[TARGET].astype(int).to_numpy()
    y_val = validation[TARGET].astype(int).to_numpy()

    for algorithm in ALGORITHMS:
        for specification, features in SPECIFICATIONS.items():
            model = create_model()
            model.fit(X(train, features), y_train)

            train_p = model.predict_proba(X(train, features))[:, 1]
            val_p = model.predict_proba(X(validation, features))[:, 1]

            for stage, y, p in [
                ("Train", y_train, train_p),
                ("Validation", y_val, val_p),
            ]:
                fit_rows.append({
                    "Rebalance_Date": rebalance_date,
                    "Algorithm": algorithm,
                    "Specification": specification,
                    "Stage": stage,
                    "Observations": len(y),
                    "Baseline_Precision": float(y.mean()),
                    "ROC_AUC": safe_auc(y, p),
                    "Average_Precision": safe_ap(y, p),
                    "Log_Loss": float(log_loss(y, p, labels=[0, 1])),
                    **{
                        f"Precision_{cutoff:.2f}": (
                            float(y[p >= cutoff].mean())
                            if np.any(p >= cutoff) else np.nan
                        )
                        for cutoff in config.CLASSIFICATION_THRESHOLDS
                    },
                    **{
                        f"Positive_Rate_{cutoff:.2f}": float(np.mean(p >= cutoff))
                        for cutoff in config.CLASSIFICATION_THRESHOLDS
                    },
                })

            part = validation[["Date", "Ticker", TARGET]].copy()
            part["Algorithm"] = algorithm
            part["Specification"] = specification
            part["Probability"] = val_p
            prediction_parts.append(part)

oof_predictions = pd.concat(prediction_parts, ignore_index=True)
train_validation_stage = pd.DataFrame(fit_rows)

assert oof_predictions["Probability"].between(0, 1).all()
assert not oof_predictions.duplicated(
    ["Date", "Ticker", "Algorithm", "Specification"]
).any()

print("Completed validation rebalances:", len(walk_forward_splits))
print("Final evaluation model fits:", len(train_validation_stage) // 2)
print("Validation predictions:", len(oof_predictions))

Tuned XGBoost walk-forward:   0%|          | 0/24 [00:00<?, ?rebalance/s]

Completed validation rebalances: 24
Final evaluation model fits: 48
Validation predictions: 2304


## 7. Training vs validation

In [7]:
keys = ["Rebalance_Date", "Algorithm", "Specification"]

train_vals = train_validation_stage.pivot(index=keys, columns="Stage")
train_vals.columns = [f"{metric}_{stage}" for metric, stage in train_vals.columns]
train_vs_validation = train_vals.reset_index()

train_vs_validation["AUC_Gap"] = (
    train_vs_validation["ROC_AUC_Train"]
    - train_vs_validation["ROC_AUC_Validation"]
)
train_vs_validation["AP_Gap"] = (
    train_vs_validation["Average_Precision_Train"]
    - train_vs_validation["Average_Precision_Validation"]
)

train_validation_summary = (
    train_vs_validation
    .groupby(["Algorithm", "Specification"], as_index=False)[[
        "ROC_AUC_Train",
        "ROC_AUC_Validation",
        "AUC_Gap",
        "Average_Precision_Train",
        "Average_Precision_Validation",
        "AP_Gap",
        "Log_Loss_Train",
        "Log_Loss_Validation",
        "Precision_0.50_Train",
        "Precision_0.50_Validation",
        "Positive_Rate_0.50_Validation",
    ]]
    .mean()
)

display(train_validation_summary.sort_values(
    "ROC_AUC_Validation", ascending=False
).round(4))

,Algorithm,Specification,ROC_AUC_Train,ROC_AUC_Validation,AUC_Gap,Average_Precision_Train,Average_Precision_Validation,AP_Gap,Log_Loss_Train,Log_Loss_Validation,Precision_0.50_Train,Precision_0.50_Validation,Positive_Rate_0.50_Validation
0,XGBoost,With Regimes,0.8294,0.7014,0.1280,0.8751,0.7579,0.1172,0.5093,0.6447,0.7700,0.6561,0.8542
1,XGBoost,Without Regimes,0.8292,0.6829,0.1463,0.8748,0.7493,0.1255,0.5094,0.6483,0.7695,0.6734,0.8411


## 8. Classification metrics and thresholds

In [8]:
direction_rows = []

# One pooled six-year result per model/specification/threshold
for (algorithm, specification), part in oof_predictions.groupby(
    ["Algorithm", "Specification"], sort=True
):
    for threshold in config.CLASSIFICATION_THRESHOLDS:
        direction_rows.append({
            "Algorithm": algorithm,
            "Specification": specification,
            "Threshold": threshold,
            **direction_metrics(part, threshold),
        })

# Always-positive benchmark, using each asset-date once
baseline_source = oof_predictions.loc[
    (oof_predictions["Algorithm"] == ALGORITHMS[0])
    & (oof_predictions["Specification"] == "Without Regimes")
]

for threshold in config.CLASSIFICATION_THRESHOLDS:
    direction_rows.append({
        "Algorithm": "AlwaysPositive",
        "Specification": "Baseline",
        "Threshold": threshold,
        **direction_metrics(
            baseline_source.assign(Probability=1.0),
            threshold
        ),
    })

direction_results = pd.DataFrame(direction_rows)

display(
    direction_results[[
        "Algorithm", "Specification", "Threshold",
        "Precision", "Always_Positive_Precision",
        "Precision_Lift_pp", "Recall", "Specificity",
        "Balanced_Accuracy", "Positive_Prediction_Rate",
        "ROC_AUC", "Average_Precision"
    ]]
    .sort_values(
        ["Threshold", "Precision"],
        ascending=[True, False]
    )
    .round(4)
)

,Algorithm,Specification,Threshold,Precision,Always_Positive_Precision,Precision_Lift_pp,Recall,Specificity,Balanced_Accuracy,Positive_Prediction_Rate,ROC_AUC,Average_Precision
0,XGBoost,With Regimes,0.50,0.6646,0.6528,1.1856,0.8697,0.1750,0.5223,0.8542,0.5980,0.7167
3,XGBoost,Without Regimes,0.50,0.6646,0.6528,1.1825,0.8564,0.1875,0.5219,0.8411,0.5837,0.6967
6,AlwaysPositive,Baseline,0.50,0.6528,0.6528,0.0000,1.0000,0.0000,0.5000,1.0000,0.5000,0.6528
1,XGBoost,With Regimes,0.55,0.6928,0.6528,3.9973,0.8005,0.3325,0.5665,0.7543,0.5980,0.7167
4,XGBoost,Without Regimes,0.55,0.6890,0.6528,3.6217,0.7660,0.3500,0.5580,0.7257,0.5837,0.6967
7,AlwaysPositive,Baseline,0.55,0.6528,0.6528,0.0000,1.0000,0.0000,0.5000,1.0000,0.5000,0.6528
2,XGBoost,With Regimes,0.60,0.7106,0.6528,5.7785,0.6888,0.4725,0.5807,0.6328,0.5980,0.7167
5,XGBoost,Without Regimes,0.60,0.7074,0.6528,5.4609,0.6622,0.4850,0.5736,0.6111,0.5837,0.6967
8,AlwaysPositive,Baseline,0.60,0.6528,0.6528,0.0000,1.0000,0.0000,0.5000,1.0000,0.5000,0.6528


## 9. Probability-threshold and Hybrid Top-K screening

In [9]:
screening_daily_rows = []

for (algorithm, specification, date), daily in oof_predictions.groupby(
    ["Algorithm", "Specification", "Date"], sort=True
):
    ranked = daily.sort_values(["Probability", "Ticker"], ascending=[False, True], kind="stable")
    baseline = float(daily[TARGET].mean())

    choices = []
    for threshold in config.CLASSIFICATION_THRESHOLDS:
        eligible = ranked.loc[ranked["Probability"] >= threshold]

        # Threshold only: select all eligible assets
        choices.append(("Probability Threshold", threshold, eligible))

        # Hybrid: select at most K eligible assets
        for k in config.TOP_K_VALUES:
            choices.append((f"Hybrid Top {k}", threshold, eligible.head(k)))

    for rule, threshold, selected in choices:
        screening_daily_rows.append({
            "Date": date,
            "Algorithm": algorithm,
            "Specification": specification,
            "Rule": rule,
            "Threshold": threshold,
            "Available_Assets": len(daily),
            "Selected_Assets": len(selected),
            "Correct_Positives": int(selected[TARGET].sum()),
            "Same_Date_Baseline": baseline,
        })

screening_daily = pd.DataFrame(screening_daily_rows)


screening_rows = []

for (algorithm, specification, rule, threshold), part in screening_daily.groupby(
    ["Algorithm", "Specification", "Rule", "Threshold"],
    dropna=False,
    sort=True
):
    screening_rows.append({
        "Algorithm": algorithm,
        "Specification": specification,
        "Rule": rule,
        "Threshold": threshold,
        **screening_summary(part),
    })

screening_results = pd.DataFrame(screening_rows)

display(
    screening_results[[
        "Algorithm",
        "Specification",
        "Rule",
        "Threshold",
        "Dates",
        "Selected_Asset_Dates",
        "Precision",
        "Matched_Date_Baseline",
        "Precision_Lift_pp",
        "Mean_Assets",
        "Min_Assets",
        "Max_Assets",
        "Cash_Only_Dates"
    ]]
    .sort_values(["Rule", "Threshold", "Precision_Lift_pp"], ascending=[True, True, False]).round(4))

# Select final screening strategy based on validation results
selection_candidates = screening_results.loc[screening_results["Mean_Assets"] >= 10].copy()

assert not selection_candidates.empty, "No strategy meets the minimum coverage."

selected_strategy = (selection_candidates.sort_values(
        ["Precision_Lift_pp", "Mean_Assets"], ascending=[False, False]).iloc[0]
)

# Asset-level results for the selected screening strategy
selected_assets = oof_predictions.loc[
    (oof_predictions["Algorithm"] == selected_strategy["Algorithm"])
    & (oof_predictions["Specification"] == selected_strategy["Specification"])
    & (oof_predictions["Probability"] >= selected_strategy["Threshold"])
].copy()

selected_assets = selected_assets.sort_values(
    ["Date", "Probability", "Ticker"],
    ascending=[True, False, True]
)

# Apply Top-K limit only if the selected rule is Hybrid
selection_rule = str(selected_strategy["Rule"])

if selection_rule.startswith("Hybrid Top"):
    k = int(selection_rule.split()[-1])
    selected_assets = selected_assets.groupby("Date", group_keys=False).head(k)

selected_assets["Predicted_Sign"] = "Positive"
selected_assets["Actual_Sign"] = np.where(
    selected_assets[TARGET] == 1, "Positive", "Negative"
)

# Summarize selections and outcomes for each rebalance date
rebalance_summary = (
    selected_assets.groupby("Date")
    .agg(
        Selected_Assets=("Ticker", "size"),
        Correct_Positives=(TARGET, "sum")
    )
    .reindex(sorted(oof_predictions["Date"].unique()), fill_value=0)
    .reset_index()
)

rebalance_summary["Precision"] = (
    rebalance_summary["Correct_Positives"]
    / rebalance_summary["Selected_Assets"].replace(0, np.nan)
)

# Combine rebalance summaries with individual asset predictions
selected_strategy_details = rebalance_summary.merge(
    selected_assets[[
        "Date", "Ticker", "Probability", "Predicted_Sign", "Actual_Sign"
    ]],
    on="Date",
    how="left",
    validate="one_to_many"
)

print("\nSELECTED SCREENING STRATEGY")
print(f"Algorithm: {selected_strategy['Algorithm']}")
print(f"Specification: {selected_strategy['Specification']}")
print(f"Rule: {selected_strategy['Rule']}")
print(f"Threshold: {selected_strategy['Threshold']:.2f}")
print(f"Precision: {selected_strategy['Precision']:.2%}")
print(f"Precision lift: {selected_strategy['Precision_Lift_pp']:.2f} pp")
print(f"Mean selected assets: {selected_strategy['Mean_Assets']:.2f}")
print(f"Min selected assets: {int(selected_strategy['Min_Assets'])}")
print(f"Max selected assets: {int(selected_strategy['Max_Assets'])}")
print(f"Cash-only dates: {int(selected_strategy['Cash_Only_Dates'])}")

,Algorithm,Specification,Rule,Threshold,Dates,Selected_Asset_Dates,Precision,Matched_Date_Baseline,Precision_Lift_pp,Mean_Assets,Min_Assets,Max_Assets,Cash_Only_Dates
0,XGBoost,With Regimes,Hybrid Top 10,0.50,24,230,0.7522,0.6413,11.0870,9.5833,0,10,1
9,XGBoost,Without Regimes,Hybrid Top 10,0.50,24,234,0.7564,0.6460,11.0399,9.7500,4,10,0
1,XGBoost,With Regimes,Hybrid Top 10,0.55,24,230,0.7522,0.6413,11.0870,9.5833,0,10,1
10,XGBoost,Without Regimes,Hybrid Top 10,0.55,24,220,0.7545,0.6468,10.7765,9.1667,0,10,2
11,XGBoost,Without Regimes,Hybrid Top 10,0.60,24,194,0.7784,0.6550,12.3389,8.0833,0,10,2
2,XGBoost,With Regimes,Hybrid Top 10,0.60,24,203,0.7833,0.6653,11.7919,8.4583,0,10,2
3,XGBoost,With Regimes,Hybrid Top 15,0.50,24,345,0.7478,0.6413,10.6522,14.3750,0,15,1
12,XGBoost,Without Regimes,Hybrid Top 15,0.50,24,349,0.7307,0.6445,8.6199,14.5417,4,15,0
4,XGBoost,With Regimes,Hybrid Top 15,0.55,24,344,0.7471,0.6417,10.5438,14.3333,0,15,1
13,XGBoost,Without Regimes,Hybrid Top 15,0.55,24,329,0.7356,0.6462,8.9349,13.7083,0,15,2



SELECTED SCREENING STRATEGY
Algorithm: XGBoost
Specification: With Regimes
Rule: Hybrid Top 15
Threshold: 0.60
Precision: 78.19%
Precision lift: 11.61 pp
Mean selected assets: 12.42
Min selected assets: 0
Max selected assets: 15
Cash-only dates: 2


## 10. Checks

In [10]:
assert oof_predictions["Date"].nunique() == len(walk_forward_splits)
assert len(hyperparameter_search_results) == N_ITER
assert oof_predictions[["Algorithm", "Specification"]].drop_duplicates().shape[0] == 2
assert oof_predictions["Probability"].between(0, 1).all()
assert len(direction_results) == 3 * len(config.CLASSIFICATION_THRESHOLDS)
assert len(screening_results) == 2 * len(config.CLASSIFICATION_THRESHOLDS) * (1 + len(config.TOP_K_VALUES))

print("All validation checks passed.")

All validation checks passed.


## 11. Export results

In [11]:
# Export validation results and selected strategy
OUTPUT_DIR = Path("classification_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_files = {
    "xgboost_hyperparameter_search.csv": hyperparameter_search_results,
    "xgboost_hyperparameter_by_rebalance.csv": search_by_rebalance,
    "xgboost_best_hyperparameters.csv": best_hyperparameters,
    "classification_metrics.csv": direction_results,
    "train_validation_metrics.csv": train_validation_summary,
    "train_validation_by_rebalance.csv": train_vs_validation,
    "screening_metrics.csv": screening_results,
    "selected_strategy_details.csv": selected_strategy_details,
}

for filename, table in csv_files.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)

# Export selected model and screening settings
rule = str(selected_strategy["Rule"])

selected_config = {
    "algorithm": str(selected_strategy["Algorithm"]),
    "specification": str(selected_strategy["Specification"]),
    "model_params": {
        **best_xgb_params,
        "reg_alpha": 1.0,
        "tree_method": "hist",
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "n_jobs": -1,
        "random_state": int(config.RANDOM_STATE),
    },
    "screening": {
        "rule": rule,
        "threshold": float(selected_strategy["Threshold"]),
        "top_k": int(rule.split()[-1]) if rule.startswith("Hybrid Top") else None,
    },
}

json_path = OUTPUT_DIR / "selected_config.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        selected_config,
        f,
        indent=4,
        default=lambda x: x.item() if isinstance(x, np.generic) else str(x),
    )

print(f"Validation results and selected configuration saved to {OUTPUT_DIR}/")

Validation results and selected configuration saved to classification_results/
